# Silent Sting: source quality profile

## tl;dr

This diagnostic scans the complete prepared source seed. It checks row and field completeness, the observed uniqueness of `patient_id`, parseable event times, future-dated arrival rows relative to the chosen as-of date, and agreement between source and timestamp-derived delay. `patient_id` is a profile candidate only; it is not used as the warehouse case key.

## Context & Methods

The file is a single CSV snapshot. `source_row_number` was added by `scripts/prepare_seed.py`; all source field values are retained as text in the seed. The notebook uses Python's standard library and streams rows instead of loading the full file into memory. Set `SILENT_STING_AS_OF_DATE` to change the reference date for future-date checks.

### Key assumptions

- Timestamps are parsed in their source timezone-free form; no timezone is inferred.
- A future arrival date is surfaced as a review item, not automatically rejected.
- This profile does not establish case-capture completeness or population incidence.

## Data

Source artifact: `seeds/raw_triage_case_rows.csv`, generated from `silent_sting_triage_data_enriched.csv` without modifying the supplied file.


In [ ]:
import csv
import json
import os
from collections import Counter
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path

cwd = Path.cwd()
project_dir = cwd if (cwd / 'seeds' / 'raw_triage_case_rows.csv').exists() else cwd.parent
source_path = project_dir / 'seeds' / 'raw_triage_case_rows.csv'
as_of_date = date.fromisoformat(os.environ.get('SILENT_STING_AS_OF_DATE', '2026-10-23'))
print({'source_path': str(source_path), 'as_of_date': as_of_date.isoformat()})

### Run full-file checks

The summary below is deliberately bounded: it reports counts and ranges rather than dumping source rows or every province value.

In [ ]:
missing = Counter()
category_counts = {name: Counter() for name in ('gender', 'bite_source_target', 'triage_level', 'local_swelling', 'muscle_paralysis_present', 'blood_coagulation_failure')}
numeric_ranges = {name: {'count': 0, 'min': None, 'max': None, 'invalid': 0} for name in ('age', 'time_since_bite_min', 'heart_rate_bpm', 'blood_pressure_systolic')}
unique_patient_ids = set()
duplicate_patient_id_rows = 0
timestamp_ranges = {name: {'min': None, 'max': None, 'invalid': 0} for name in ('bite_timestamp', 'arrival_timestamp')}
rows = 0
arrival_before_bite_rows = 0
future_bite_rows = 0
future_arrival_rows = 0
delay_rows_compared = 0
delay_mismatch_rows = 0

with source_path.open('r', newline='', encoding='utf-8-sig') as source:
    reader = csv.DictReader(source)
    columns = reader.fieldnames
    for row in reader:
        rows += 1
        for name, value in row.items():
            if value is None or not value.strip():
                missing[name] += 1
        patient_id = (row.get('patient_id') or '').strip()
        if patient_id in unique_patient_ids:
            duplicate_patient_id_rows += 1
        unique_patient_ids.add(patient_id)
        for name, counts in category_counts.items():
            counts[(row.get(name) or '').strip()] += 1
        for name, stats in numeric_ranges.items():
            value = (row.get(name) or '').strip()
            if not value:
                continue
            try:
                number = Decimal(value)
            except Exception:
                stats['invalid'] += 1
                continue
            stats['count'] += 1
            stats['min'] = number if stats['min'] is None else min(stats['min'], number)
            stats['max'] = number if stats['max'] is None else max(stats['max'], number)
        parsed = {}
        for name, stats in timestamp_ranges.items():
            value = (row.get(name) or '').strip()
            try:
                timestamp = datetime.strptime(value, '%Y-%m-%d %H:%M:%S')
            except ValueError:
                stats['invalid'] += 1
                continue
            parsed[name] = timestamp
            stats['min'] = timestamp if stats['min'] is None else min(stats['min'], timestamp)
            stats['max'] = timestamp if stats['max'] is None else max(stats['max'], timestamp)
            if timestamp.date() > as_of_date:
                if name == 'bite_timestamp':
                    future_bite_rows += 1
                else:
                    future_arrival_rows += 1
        if len(parsed) == 2:
            calculated_minutes = Decimal(str((parsed['arrival_timestamp'] - parsed['bite_timestamp']).total_seconds())) / Decimal(60)
            if calculated_minutes < 0:
                arrival_before_bite_rows += 1
            source_minutes = (row.get('time_since_bite_min') or '').strip()
            if source_minutes:
                delay_rows_compared += 1
                if Decimal(source_minutes) != calculated_minutes:
                    delay_mismatch_rows += 1

profile = {
    'row_count': rows,
    'column_count': len(columns or []),
    'duplicate_patient_id_rows': duplicate_patient_id_rows,
    'missing_by_column': dict(missing),
    'numeric_ranges': {name: {key: (str(value) if isinstance(value, Decimal) else value) for key, value in stats.items()} for name, stats in numeric_ranges.items()},
    'timestamp_ranges': {name: {key: (value.isoformat(sep=' ') if isinstance(value, datetime) else value) for key, value in stats.items()} for name, stats in timestamp_ranges.items()},
    'future_bite_rows': future_bite_rows,
    'future_arrival_rows': future_arrival_rows,
    'arrival_before_bite_rows': arrival_before_bite_rows,
    'delay_rows_compared': delay_rows_compared,
    'delay_mismatch_rows': delay_mismatch_rows,
    'category_counts': {name: dict(counts) for name, counts in category_counts.items()},
}
print(json.dumps(profile, ensure_ascii=False, indent=2))

## Results

The output above is the measured profile for the complete seed. In particular, check the future-arrival count against the documented reporting cutoff before presenting calendar trends as observed history.

## Takeaways

- Keep `patient_id` out of the general fact/reporting layer; uniqueness in one file does not establish person-versus-encounter semantics.
- Preserve the source and timestamp-derived delay values separately; do not use one to overwrite the other.
- Confirm why the source includes event dates beyond the current as-of date. Those rows remain in the prototype unless the data owner sets a rule.
- This check measures file structure and internal consistency only; it does not verify coverage, denominator suitability, or clinical meaning.